# Topic 22 — Bag of Words
### Theory → from-scratch implementation → sklearn CountVectorizer → sparsity.

**Bag of Words (BoW)** is the simplest way to turn text into numeric features: count how many times
each word appears, ignoring grammar and word order entirely (hence "bag" — just a jumble of words).

- **Vocabulary**: the set of all unique words across your documents.
- **Token**: one unit of text (usually a word) after tokenization.
- **Document**: one piece of text (e.g. one social media post/comment).
- **Term frequency**: how many times a word appears in a document.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

docs = [
    "you are stupid and worthless",
    "i hate you so much",
    "great job today team",
    "you are stupid",
]

## 1. From-scratch Bag of Words

Build the vocabulary, then count each document's words against it.

In [ ]:
def build_vocabulary(documents):
    vocab = set()
    for doc in documents:
        vocab.update(doc.lower().split())
    return sorted(vocab)   # sorted for a consistent, reproducible column order

def documents_to_bow(documents, vocab):
    matrix = np.zeros((len(documents), len(vocab)), dtype=int)
    word_to_idx = {word: i for i, word in enumerate(vocab)}
    for row, doc in enumerate(documents):
        for word in doc.lower().split():
            if word in word_to_idx:
                matrix[row, word_to_idx[word]] += 1
    return matrix

vocab = build_vocabulary(docs)
print("vocabulary:", vocab)
print("vocabulary size:", len(vocab))

bow_matrix = documents_to_bow(docs, vocab)
bow_df = pd.DataFrame(bow_matrix, columns=vocab)
print(bow_df)

## 2. sklearn's CountVectorizer — the same thing, production-ready

In [ ]:
vectorizer = CountVectorizer()
X_counts = vectorizer.fit_transform(docs)

print("vocabulary:", vectorizer.get_feature_names_out())
print("shape:", X_counts.shape, " (n_documents, vocabulary_size)")

# .toarray() converts the sparse matrix to a normal dense NumPy array for viewing
print(pd.DataFrame(X_counts.toarray(), columns=vectorizer.get_feature_names_out()))
# Compare this to your from-scratch matrix above -- should match (aside from column order).

## 3. Sparse matrices — why they matter

Real vocabularies have thousands of words, but any single short document only uses a handful of
them. Storing every zero explicitly would waste enormous memory. A **sparse matrix** only stores
the *non-zero* values and their positions — CountVectorizer returns this format by default.

In [ ]:
print("type of X_counts:", type(X_counts))
print("stored (non-zero) values:", X_counts.nnz)
print("total cells if stored densely:", X_counts.shape[0] * X_counts.shape[1])
print("sparsity: {:.1f}% of cells are zero".format(
    100 * (1 - X_counts.nnz / (X_counts.shape[0] * X_counts.shape[1]))
))

# Illustrate with a bigger, more realistic vocabulary
bigger_docs = docs + [
    "the quick brown fox jumps over the lazy dog",
    "machine learning models need clean data",
    "cyberbullying detection is an important research problem",
]
bigger_vec = CountVectorizer()
X_bigger = bigger_vec.fit_transform(bigger_docs)
print("\nwith more documents/vocabulary:")
print("shape:", X_bigger.shape)
print("sparsity: {:.1f}%".format(100 * (1 - X_bigger.nnz / (X_bigger.shape[0] * X_bigger.shape[1]))))
# Sparsity typically climbs above 90% quickly as vocabulary grows -- this is why sparse storage
# (and models that handle sparse input well, like Naive Bayes and linear SVM) matter for text.

## 4. Useful CountVectorizer options

In [ ]:
# min_df: ignore words that appear in fewer than N documents (removes rare noise/typos)
# max_df: ignore words that appear in MORE than this fraction of documents (removes overly common words)
vec_filtered = CountVectorizer(min_df=1, max_df=0.9, stop_words="english")
X_filtered = vec_filtered.fit_transform(bigger_docs)
print("vocabulary after filtering:", vec_filtered.get_feature_names_out())

# max_features: cap the vocabulary to the top-N most frequent words -- keeps feature count manageable
vec_capped = CountVectorizer(max_features=10)
X_capped = vec_capped.fit_transform(bigger_docs)
print("\ntop-10 vocabulary:", vec_capped.get_feature_names_out())

## 5. Important: `.transform()` on new/test data, never re-fit

Exactly like scalers/imputers (Topics 15-17) — the vectorizer's vocabulary must be learned ONLY on
training data, then applied unchanged to test data. Any word in test data that wasn't seen during
training is simply ignored (not added as a new column).

In [ ]:
train_docs = ["you are stupid", "great job today"]
test_docs = ["you are wonderful and amazing"]   # contains brand-new words never seen in train

vec = CountVectorizer().fit(train_docs)
X_train = vec.transform(train_docs)
X_test = vec.transform(test_docs)   # NOT fit_transform!

print("train vocabulary:", vec.get_feature_names_out())
print("test doc as vector:", X_test.toarray())
# Notice "wonderful" and "amazing" simply don't appear -- the vector only reflects words
# that were part of the TRAINING vocabulary.

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Build the from-scratch BoW matrix for 3 of your own sentences and verify it by hand.
# 2. Use CountVectorizer with binary=True (presence/absence instead of counts) and compare the
#    resulting matrix to the default count-based one -- this is what BernoulliNB (Topic 11) expects.
# 3. Try min_df=2 on `bigger_docs` -- which words get dropped for being too rare?
# 4. Explain in one sentence why Bag of Words loses information that word ORDER would carry
#    (e.g. "not good" vs "good not" produce identical BoW vectors) -- this motivates n-grams next.

---
### Next up: **Topic 23 — N-grams** (partially recovering word order/context).

Say "next" when you're ready.